In [1]:
import re, random, numpy as np, pandas as pd
random.seed(42); np.random.seed(42)

ventas = [
    "Quiero saber el precio del plan premium",
    "¿Tienen descuentos por volumen para empresas?",
    "¿Cómo puedo pagar? ¿Tarjeta o transferencia?",
    "Estoy interesado en comprar 10 unidades",
    "¿Cuánto cuesta el plan anual y cómo se factura?"
]
soporte = [
    "No puedo iniciar sesión, sale error 403",
    "La app se cierra al abrir el carrito",
    "La impresora no conecta por wifi, ya reinicié",
    "Se perdió mi pedido en la app, ayuda",
    "No me llega el código de verificación"
]
queja = [
    "El pedido llegó incompleto y nadie responde",
    "Muy mala atención, llegó tarde y mal empacado",
    "Estoy inconforme, el producto vino dañado",
    "Demasiada demora, pésimo servicio",
    "Me trataron mal por WhatsApp, muy groseros"
]

def variar(s):
    extras = ["", "!", "!!", " por favor", " urgente", " de verdad", " gracias"]
    return s + random.choice(extras)

data = []
for _ in range(20):
    data += [(variar(x), "ventas") for x in ventas]
    data += [(variar(x), "soporte") for x in soporte]
    data += [(variar(x), "queja")   for x in queja]

df = pd.DataFrame(data, columns=["texto","etiqueta"]).sample(frac=1, random_state=42).reset_index(drop=True)
print("Muestras:", len(df), df["etiqueta"].value_counts().to_dict())

Muestras: 300 {'soporte': 100, 'queja': 100, 'ventas': 100}


In [2]:
import re

def limpiar(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-záéíóúñü0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["texto_clean"] = df["texto"].apply(limpiar)

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["texto_clean"], df["etiqueta"], test_size=0.2, random_state=42, stratify=df["etiqueta"]
)

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

pipe = make_pipeline(
    TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=2),
    LinearSVC(class_weight="balanced", random_state=42)
)

In [5]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

acc = accuracy_score(y_test, pred)
print(f"\nAccuracy test: {acc:.3f}\n")
print("Reporte por clase:\n", classification_report(y_test, pred, digits=3))

cm = confusion_matrix(y_test, pred, labels=["ventas","soporte","queja"])
print("\nMatriz de confusión (filas=real, cols=pred):\n", pd.DataFrame(cm,
      index=["real_ventas","real_soporte","real_queja"],
      columns=["pred_ventas","pred_soporte","pred_queja"]))


Accuracy test: 1.000

Reporte por clase:
               precision    recall  f1-score   support

       queja      1.000     1.000     1.000        20
     soporte      1.000     1.000     1.000        20
      ventas      1.000     1.000     1.000        20

    accuracy                          1.000        60
   macro avg      1.000     1.000     1.000        60
weighted avg      1.000     1.000     1.000        60


Matriz de confusión (filas=real, cols=pred):
               pred_ventas  pred_soporte  pred_queja
real_ventas            20             0           0
real_soporte            0            20           0
real_queja              0             0          20


In [6]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe, df["texto_clean"], df["etiqueta"], cv=5, scoring="f1_macro")
print(f"\nCV 5-fold F1_macro: media={scores.mean():.3f} ±{scores.std():.3f}")


CV 5-fold F1_macro: media=1.000 ±0.000


In [7]:
def enrutar_mensajes(textos):
    tx = [limpiar(t) for t in textos]
    etiquetas = pipe.predict(tx)
    area = {"ventas":"Equipo Ventas", "soporte":"Mesa Soporte", "queja":"Atención al Cliente"}
    rutas = [area[e] for e in etiquetas]
    return list(zip(textos, etiquetas, rutas))

nuevos = [
    "Se dañó el botón de encendido, necesito ayuda urgentemente",
    "¿Hacen descuento si compro 15 licencias?",
    "Estoy muy molesto: llegó tarde y la caja rota, pésimo servicio"
]
print("\nEnrutamiento de mensajes nuevos:")
for texto, etiqueta, ruta in enrutar_mensajes(nuevos):
    print(f"- '{texto}' -> clase: {etiqueta} | ruta: {ruta}")


Enrutamiento de mensajes nuevos:
- 'Se dañó el botón de encendido, necesito ayuda urgentemente' -> clase: soporte | ruta: Mesa Soporte
- '¿Hacen descuento si compro 15 licencias?' -> clase: queja | ruta: Atención al Cliente
- 'Estoy muy molesto: llegó tarde y la caja rota, pésimo servicio' -> clase: queja | ruta: Atención al Cliente


In [8]:
import joblib

joblib.dump(pipe, "pipeline_triage.joblib")
print("\nPipeline guardado en pipeline_triage.joblib")

loaded = joblib.load("pipeline_triage.joblib")
print("Test carga:", loaded.predict(["No puedo entrar a mi cuenta, sale error 500"])[0])


Pipeline guardado en pipeline_triage.joblib
Test carga: soporte


In [ ]:
# df_real = pd.read_csv("mensajes.csv")  # columnas: id, texto
# df_real["texto_clean"] = df_real["texto"].apply(limpiar)
# df_real["etiqueta"] = loaded.predict(df_real["texto_clean"])
# df_real["ruta"] = df_real["etiqueta"].map({"ventas":"Equipo Ventas","soporte":"Mesa Soporte","queja":"Atención al Cliente"})
# df_real.to_csv("mensajes_enrutados.csv", index=False)
# print("Archivo 'mensajes_enrutados.csv' generado con la ruta de cada mensaje.")